# Cohort Recompute - PIP-61

Computes cohort statistics for the two new OECD indicators added in feature
branch `feat/oecd-indicator-completeness`:

- `patent_scope` (CPC subclass count, OECD §3.3)
- `radicalness_index` (share of backward citations in different CPC subclass, OECD §3.7)

Year range matches the existing `cohort-stats.json`: 1978-2024.
WIPO field assignment: primary tech field per patent (highest weight in `tls230_appln_techn_field`).

**Outputs (alongside this notebook, gitignored):**
- `cohort-stats-additions.json` - rows to append to `src/lib/server/data/cohort-stats.json`
- `reference-patents-patches.json` - per-publication patches for `src/lib/server/data/reference-patents.json`

**Runtime:** Patent Scope ~2 min, Radicalness ~10-30 min depending on PATSTAT load.

**Run on:** EPO TIP (uses `epo.tipdata.patstat` directly, BigQuery is included).

In [ ]:
import json
from pathlib import Path

from epo.tipdata.patstat import PatstatClient

client = PatstatClient(env='PROD')
db = client.orm()

OUT_DIR = Path.cwd()
YEAR_MIN, YEAR_MAX = 1978, 2024

# Locate the cloned repo. The launch notebook clones it to ~/patent_value_explorer.
REPO_CANDIDATES = [
    Path.cwd() / 'src' / 'lib' / 'server' / 'data',
    Path.home() / 'patent_value_explorer' / 'src' / 'lib' / 'server' / 'data',
]
REPO_DATA = next((p for p in REPO_CANDIDATES if (p / 'reference-patents.json').exists()), None)
if REPO_DATA is None:
    raise FileNotFoundError(
        'reference-patents.json not found. Tried:\n  '
        + '\n  '.join(str(p) for p in REPO_CANDIDATES)
        + '\nClone the repo first via the Patent Value Explorer launch notebook,'
        + ' or set REPO_DATA manually.'
    )

REFS = json.loads((REPO_DATA / 'reference-patents.json').read_text())

print(f'PATSTAT client ready. {len(REFS)} reference patents loaded from {REPO_DATA}')
print(f'Year range: {YEAR_MIN}-{YEAR_MAX}')

## WIPO field name mapping

Cohort rows include both `wipoFieldNumber` (1-35) and `wipoFieldName`.
Pull the canonical name once from `tls901_techn_field_ipc`.

In [ ]:
from epo.tipdata.patstat.database.models import TLS901_TECHN_FIELD_IPC
from sqlalchemy import distinct

rows = db.query(
    distinct(TLS901_TECHN_FIELD_IPC.techn_field_nr),
    TLS901_TECHN_FIELD_IPC.techn_field
).all()

WIPO_NAMES = {int(nr): name for nr, name in rows}
print(f'{len(WIPO_NAMES)} WIPO fields:')
for nr in sorted(WIPO_NAMES):
    print(f'  {nr:2d}  {WIPO_NAMES[nr]}')

## 1. Patent Scope cohort statistics

Per-patent: `COUNT(DISTINCT SUBSTR(cpc_class_symbol, 1, 4))` (number of CPC subclasses).
Per-cohort: `count`, `mean`, `median`, `p1/p5/p25/p75/p95/p99`, `max`.
Cohort = (`primary_wipo_field`, `filing_year`).

In [ ]:
from sqlalchemy import text

SQL_PATENT_SCOPE = text(f"""
WITH primary_field AS (
  SELECT appln_id, techn_field_nr,
         ROW_NUMBER() OVER (
           PARTITION BY appln_id
           ORDER BY weight DESC, techn_field_nr ASC
         ) AS rn
  FROM tls230_appln_techn_field
),
patent_scope AS (
  SELECT
    a.appln_id,
    EXTRACT(YEAR FROM a.appln_filing_date) AS filing_year,
    pf.techn_field_nr AS wipo_field,
    COUNT(DISTINCT SUBSTR(c.cpc_class_symbol, 1, 4)) AS scope
  FROM tls201_appln a
  JOIN tls224_appln_cpc c ON c.appln_id = a.appln_id
  JOIN primary_field pf ON pf.appln_id = a.appln_id AND pf.rn = 1
  WHERE EXTRACT(YEAR FROM a.appln_filing_date) BETWEEN {YEAR_MIN} AND {YEAR_MAX}
  GROUP BY a.appln_id, filing_year, wipo_field
)
SELECT
  wipo_field AS wipo_field_number,
  filing_year AS filing_year,
  COUNT(*) AS count,
  AVG(scope) AS mean,
  APPROX_QUANTILES(scope, 100)[OFFSET(50)] AS median,
  APPROX_QUANTILES(scope, 100)[OFFSET(1)]  AS p1,
  APPROX_QUANTILES(scope, 100)[OFFSET(5)]  AS p5,
  APPROX_QUANTILES(scope, 100)[OFFSET(25)] AS p25,
  APPROX_QUANTILES(scope, 100)[OFFSET(75)] AS p75,
  APPROX_QUANTILES(scope, 100)[OFFSET(95)] AS p95,
  APPROX_QUANTILES(scope, 100)[OFFSET(99)] AS p99,
  MAX(scope) AS max
FROM patent_scope
GROUP BY wipo_field_number, filing_year
HAVING COUNT(*) > 0
ORDER BY wipo_field_number, filing_year
""")

print('Running Patent Scope cohort query...')
rows = db.execute(SQL_PATENT_SCOPE).all()
print(f'{len(rows)} cohort rows')

patent_scope_cohorts = [
    {
        'wipoFieldNumber': int(r.wipo_field_number),
        'wipoFieldName': WIPO_NAMES.get(int(r.wipo_field_number), 'Unknown'),
        'filingYear': int(r.filing_year),
        'indicator': 'patent_scope',
        'count': int(r.count),
        'mean': float(r.mean),
        'median': float(r.median),
        'p1': float(r.p1),
        'p5': float(r.p5),
        'p25': float(r.p25),
        'p75': float(r.p75),
        'p95': float(r.p95),
        'p99': float(r.p99),
        'max': float(r.max),
    }
    for r in rows
]
print(f'Sample: {patent_scope_cohorts[0]}')

## 2. Radicalness cohort statistics

Per-patent: share of backward citations whose CPC subclasses do NOT overlap with focal patent's CPC subclasses.

  RAD = COUNTIF(focal_subclasses INTERSECT cited_subclasses = empty) / total_backward_citations

Heavy query - joins `tls201 -> tls211 -> tls212 -> tls211 (cited) -> tls224 (cited cpc) -> tls224 (focal cpc)`.
Uses ARRAY_AGG to materialize subclass sets per patent and `ARRAY_LENGTH(ARRAY_INTERSECT(...))` for overlap.

If this times out on TIP, see the per-WIPO-field loop variant below as a fallback.

In [ ]:
SQL_RADICALNESS = text(f"""
WITH primary_field AS (
  SELECT appln_id, techn_field_nr,
         ROW_NUMBER() OVER (
           PARTITION BY appln_id
           ORDER BY weight DESC, techn_field_nr ASC
         ) AS rn
  FROM tls230_appln_techn_field
),
focal_subs AS (
  SELECT a.appln_id,
         EXTRACT(YEAR FROM a.appln_filing_date) AS filing_year,
         pf.techn_field_nr AS wipo_field,
         ARRAY_AGG(DISTINCT SUBSTR(c.cpc_class_symbol, 1, 4)) AS focal_subs
  FROM tls201_appln a
  JOIN tls224_appln_cpc c ON c.appln_id = a.appln_id
  JOIN primary_field pf ON pf.appln_id = a.appln_id AND pf.rn = 1
  WHERE EXTRACT(YEAR FROM a.appln_filing_date) BETWEEN {YEAR_MIN} AND {YEAR_MAX}
  GROUP BY a.appln_id, filing_year, wipo_field
),
cited_subs AS (
  SELECT p.appln_id AS focal_appln,
         cited_pub.appln_id AS cited_appln,
         ARRAY_AGG(DISTINCT SUBSTR(cited_cpc.cpc_class_symbol, 1, 4)) AS cited_subs
  FROM tls211_pat_publn p
  JOIN tls212_citation c ON c.pat_publn_id = p.pat_publn_id
  JOIN tls211_pat_publn cited_pub ON cited_pub.pat_publn_id = c.cited_pat_publn_id
  JOIN tls224_appln_cpc cited_cpc ON cited_cpc.appln_id = cited_pub.appln_id
  WHERE c.cited_pat_publn_id > 0
  GROUP BY focal_appln, cited_appln
),
radical_per_patent AS (
  SELECT f.appln_id,
         f.filing_year,
         f.wipo_field,
         COUNT(*) AS total_cited,
         COUNTIF(
           (SELECT COUNT(*) FROM UNNEST(cs.cited_subs) sub
            WHERE sub IN UNNEST(f.focal_subs)) = 0
         ) AS radical_cited
  FROM focal_subs f
  JOIN cited_subs cs ON cs.focal_appln = f.appln_id
  GROUP BY f.appln_id, f.filing_year, f.wipo_field
),
rad_score AS (
  SELECT appln_id, filing_year, wipo_field,
         SAFE_DIVIDE(radical_cited, total_cited) AS rad
  FROM radical_per_patent
  WHERE total_cited > 0
)
SELECT
  wipo_field AS wipo_field_number,
  filing_year AS filing_year,
  COUNT(*) AS count,
  AVG(rad) AS mean,
  APPROX_QUANTILES(rad, 100)[OFFSET(50)] AS median,
  APPROX_QUANTILES(rad, 100)[OFFSET(1)]  AS p1,
  APPROX_QUANTILES(rad, 100)[OFFSET(5)]  AS p5,
  APPROX_QUANTILES(rad, 100)[OFFSET(25)] AS p25,
  APPROX_QUANTILES(rad, 100)[OFFSET(75)] AS p75,
  APPROX_QUANTILES(rad, 100)[OFFSET(95)] AS p95,
  APPROX_QUANTILES(rad, 100)[OFFSET(99)] AS p99,
  MAX(rad) AS max
FROM rad_score
GROUP BY wipo_field_number, filing_year
HAVING COUNT(*) > 0
ORDER BY wipo_field_number, filing_year
""")

print('Running Radicalness cohort query (heavier - allow several minutes)...')
rows = db.execute(SQL_RADICALNESS).all()
print(f'{len(rows)} cohort rows')

radicalness_cohorts = [
    {
        'wipoFieldNumber': int(r.wipo_field_number),
        'wipoFieldName': WIPO_NAMES.get(int(r.wipo_field_number), 'Unknown'),
        'filingYear': int(r.filing_year),
        'indicator': 'radicalness_index',
        'count': int(r.count),
        'mean': float(r.mean),
        'median': float(r.median),
        'p1': float(r.p1),
        'p5': float(r.p5),
        'p25': float(r.p25),
        'p75': float(r.p75),
        'p95': float(r.p95),
        'p99': float(r.p99),
        'max': float(r.max),
    }
    for r in rows
]
print(f'Sample: {radicalness_cohorts[0] if radicalness_cohorts else None}')

## 3. Write cohort additions JSON

Output is appended to `src/lib/server/data/cohort-stats.json` after committing
(use `jq -s '.[0] + .[1]' src/lib/server/data/cohort-stats.json cohort-stats-additions.json > merged.json`).

In [ ]:
all_new = patent_scope_cohorts + radicalness_cohorts
out_path = OUT_DIR / 'cohort-stats-additions.json'
out_path.write_text(json.dumps(all_new, indent='\t'))
print(f'Wrote {len(all_new)} rows to {out_path}')
print(f'  patent_scope:      {len(patent_scope_cohorts)} rows')
print(f'  radicalness_index: {len(radicalness_cohorts)} rows')

## 4. Reference patents backfill

For each patent in `reference-patents.json`, compute `patentScope` and `radicalnessIndex`
live, then normalize against the new cohort stats.

In [ ]:
def cohort_lookup(cohorts, indicator, wipo, year):
    for c in cohorts:
        if c['indicator'] == indicator and c['wipoFieldNumber'] == wipo and c['filingYear'] == year:
            return c
    return None

def winsorize(value, p1, p99):
    return max(p1, min(p99, value))

def normalize(value, cohort):
    if cohort is None or value is None:
        return None
    if cohort['p99'] == cohort['p1']:
        return 0.5
    w = winsorize(value, cohort['p1'], cohort['p99'])
    return round((w - cohort['p1']) / (cohort['p99'] - cohort['p1']), 4)


# BigQuery's SQLAlchemy dialect doesn't always accept :name binds reliably for
# integer columns - inline the appln_id directly (it's a trusted int from our
# own JSON) to avoid 'parameter type unknown' hangs.
def sql_patent_scope(aid: int) -> str:
    return f"""
    SELECT COUNT(DISTINCT SUBSTR(cpc_class_symbol, 1, 4)) AS scope
    FROM tls224_appln_cpc
    WHERE appln_id = {int(aid)}
    """

def sql_radicalness(aid: int) -> str:
    # Note: column and CTE must not share a name in BigQuery, otherwise the
    # outer `COUNTIF(col = 0)` is parsed as `COUNTIF(table_struct = 0)`.
    return f"""
    WITH focal AS (
      SELECT DISTINCT SUBSTR(cpc_class_symbol, 1, 4) AS sub
      FROM tls224_appln_cpc WHERE appln_id = {int(aid)}
    ),
    cited_subs AS (
      SELECT cited_pub.appln_id AS cited_aid,
             SUBSTR(cited_cpc.cpc_class_symbol, 1, 4) AS sub
      FROM tls211_pat_publn p
      JOIN tls212_citation c ON c.pat_publn_id = p.pat_publn_id
      JOIN tls211_pat_publn cited_pub ON cited_pub.pat_publn_id = c.cited_pat_publn_id
      JOIN tls224_appln_cpc cited_cpc ON cited_cpc.appln_id = cited_pub.appln_id
      WHERE p.appln_id = {int(aid)} AND c.cited_pat_publn_id > 0
    ),
    overlap_per_cited AS (
      SELECT cs.cited_aid, COUNTIF(f.sub IS NOT NULL) AS overlap_cnt
      FROM cited_subs cs
      LEFT JOIN focal f ON f.sub = cs.sub
      GROUP BY cs.cited_aid
    )
    SELECT COUNT(*) AS total, COUNTIF(overlap_cnt = 0) AS radical
    FROM overlap_per_cited
    """

In [ ]:
import time
from sqlalchemy import text

db.rollback()  # clear any leftover failed-transaction state

patches = []
for i, ref in enumerate(REFS, start=1):
    aid = int(ref['applnId'])
    wipo = int(ref['wipoFieldNumber'])
    year = int(ref['filingDate'][:4])
    pub = ref['publicationNumber']
    print(f'[{i}/{len(REFS)}] {pub:<14}  appln_id={aid}  wipo={wipo}  year={year}', flush=True)

    # Patent Scope
    t0 = time.time()
    try:
        row = db.execute(text(sql_patent_scope(aid))).first()
        scope = int(row.scope) if row and row.scope else None
        if scope == 0:
            scope = None
    except Exception as e:
        print(f'  patent_scope FAILED: {e}', flush=True)
        db.rollback()
        scope = None
    scope_norm = normalize(scope, cohort_lookup(patent_scope_cohorts, 'patent_scope', wipo, year))
    print(f'  patent_scope: {scope} (normalized={scope_norm}) in {time.time()-t0:.1f}s', flush=True)

    # Radicalness
    t0 = time.time()
    try:
        row = db.execute(text(sql_radicalness(aid))).first()
        if row and row.total and row.total > 0:
            rad = round(row.radical / row.total, 4)
        else:
            rad = None
    except Exception as e:
        print(f'  radicalness FAILED: {e}', flush=True)
        db.rollback()
        rad = None
    rad_norm = normalize(rad, cohort_lookup(radicalness_cohorts, 'radicalness_index', wipo, year))
    print(f'  radicalness:  {rad} (normalized={rad_norm}) in {time.time()-t0:.1f}s', flush=True)

    patches.append({
        'publicationNumber': pub,
        'patentScope': scope,
        'patentScopeNormalized': scope_norm,
        'radicalnessIndex': rad,
        'radicalnessIndexNormalized': rad_norm,
    })

out_path = OUT_DIR / 'reference-patents-patches.json'
out_path.write_text(json.dumps(patches, indent='\t'))
print(f'\nWrote {len(patches)} patches to {out_path}', flush=True)

## 5. Merge into repo (manual, on dev machine)

Copy `cohort-stats-additions.json` and `reference-patents-patches.json` off TIP, then on your
dev machine:

```bash
# Append new cohort rows
jq -s '.[0] + .[1]' \
  src/lib/server/data/cohort-stats.json \
  cohort-stats-additions.json \
  > tmp.json && mv tmp.json src/lib/server/data/cohort-stats.json

# Patch reference patents
python3 - <<'PY'
import json
refs = json.load(open('src/lib/server/data/reference-patents.json'))
patches = {p['publicationNumber']: p for p in json.load(open('reference-patents-patches.json'))}
for r in refs:
    p = patches.get(r['publicationNumber'])
    if p:
        r['patentScope'] = p['patentScope']
        r['patentScopeNormalized'] = p['patentScopeNormalized']
        r['radicalnessIndex'] = p['radicalnessIndex']
        r['radicalnessIndexNormalized'] = p['radicalnessIndexNormalized']
open('src/lib/server/data/reference-patents.json', 'w').write(
    json.dumps(refs, indent='\t')
)
PY
```

Commit with message `PIP-61 data: backfill patent_scope and radicalness_index cohorts and references`.